# Sequential Convex Programming for Trajectory Generation

### Libraries

In [ ]:
SOLVER_BACKEND = "icore_cpp"  # cvxpy_qoco, qoco_native, jax_icore, icore_c, icore_cpp
DYNAMICS_AD = "forward"       # forward or reverse
CT_AD = "auto"                # auto, forward or reverse
SOLVER_TOLERANCE = 1e-8
QOCO_OPTIONS = dict(abstol=SOLVER_TOLERANCE, reltol=0.0,
                    abstol_inacc=SOLVER_TOLERANCE, reltol_inacc=0.0,
                    max_iters=200)
EAGER_COMPILE = False  # Keep inner JIT functions composable in SCP kernels.

DISCRETIZATION_MODE = "fast"  # native CPU when certified; jax (previous fast), reference, or native


Timing columns are milliseconds. Native build and JIT compilation occur before iteration timing.

In [ ]:
%load_ext autoreload
%autoreload 2

from jax import config
config.update("jax_enable_x64", True)

import os
import copy
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt


from mpc import RUN
from scp import prox_convex
from cvx import parse_convex_problem
from disc import jit_int_mult_fcn, jit_cal_disc_fcn
from jitting import jit_ncvx_dt_fcn_grad, jit_ncvx_ct_fcn_grad, jit_ncvx_cvx_fcn_grad, jit_ncvx_smth_fcn_grad, jit_ncvx_comp_fcn_grad

from utils import profile_with_cprofile, make_anim

from model.di_eventually.dynamics import dynamics
from model.di_eventually.params import params_fcn, scale_params, unscale_prox_results
from model.di_eventually.cost_cons import cvx_cost_fcn, cvx_cons_fcn, ncvx_dt_fcn, ncvx_ct_fcn, ncvx_cvx_fcn, ncvx_cvx_cp_fcn, ncvx_smth_fcn, ncvx_comp_fcn
from model.di_eventually.plot import plot_eventually_demo_figure, save_individual_eventually_figures



### Parameters

In [ ]:
K = 7 # Total number of nodes
t_f = 12.0 # Total simulation time [s] spd = 1

free_final_time = False
time_dil = False

params = params_fcn(t_f, K, free_final_time, time_dil)

params['generate_code'] = False
params['use_generated_code'] = False
params['cvxpy_solver'] = 'QOCO'
params['qoco_options'] = QOCO_OPTIONS.copy()
params['dynamics_autodiff'] = DYNAMICS_AD
params['discretization_mode'] = DISCRETIZATION_MODE

params_sc = scale_params(copy.deepcopy(params))
params_sc = dynamics(copy.deepcopy(params_sc))

### Call functions

In [ ]:
fcn_dict = {}

fcn_dict['cvx_cost_fcn'] = cvx_cost_fcn
fcn_dict['cvx_cons_fcn'] = cvx_cons_fcn

In [ ]:
fcn_dict['cal_disc_jitted'] = jit_cal_disc_fcn(params_sc, compile=EAGER_COMPILE)
fcn_dict['int_mult_jitted'] = jit_int_mult_fcn(params_sc, compile=EAGER_COMPILE)

In [ ]:
fcn_dict['ncvx_dt_fcn_jitted'], fcn_dict['ncvx_dt_fcn_grad_jitted'] = jit_ncvx_dt_fcn_grad(ncvx_dt_fcn, params_sc, compile=EAGER_COMPILE)
fcn_dict['ncvx_ct_fcn_jitted'], fcn_dict['ncvx_ct_fcn_grad_jitted'] = jit_ncvx_ct_fcn_grad(ncvx_ct_fcn, params_sc, autodiff=CT_AD, compile=EAGER_COMPILE)

In [ ]:
# -------------------------------------------
# --------------- prox-convex ---------------
# -------------------------------------------
fcn_dict['ncvx_cvx_fcn'] = ncvx_cvx_fcn
fcn_dict['ncvx_cvx_cp_fcn'] = ncvx_cvx_cp_fcn

fcn_dict['ncvx_cvx_fcn_jitted'], fcn_dict['ncvx_cvx_fcn_grad_jitted'] = jit_ncvx_cvx_fcn_grad(ncvx_cvx_fcn, params_sc, compile=EAGER_COMPILE)
fcn_dict['ncvx_smth_fcn_jitted'], fcn_dict['ncvx_smth_fcn_grad_jitted'] = jit_ncvx_smth_fcn_grad(ncvx_smth_fcn, params_sc, compile=EAGER_COMPILE)
fcn_dict['ncvx_comp_fcn_jitted'], fcn_dict['ncvx_comp_fcn_grad_jitted'] = jit_ncvx_comp_fcn_grad(ncvx_comp_fcn, params_sc, compile=EAGER_COMPILE)
# -------------------------------------------
# --------------- prox-convex ---------------
# -------------------------------------------

### Convex solver

In [ ]:
if SOLVER_BACKEND == "cvxpy_qoco":
    cvx_prb = parse_convex_problem(params_sc, fcn_dict)
else:
    from ipm_adapter import OptionalConvexProblem
    cvx_prb = OptionalConvexProblem(
        params_sc, fcn_dict, SOLVER_BACKEND,
        tolerance=SOLVER_TOLERANCE, qoco_options=QOCO_OPTIONS,
    )
    cvx_prb.prepare(fcn_dict)
    print(f"Solver/model JIT compilation: {1000.0 * (cvx_prb.compile_seconds):.3f} ms")
    print(f"Native build/workspace setup: {1000.0 * (cvx_prb.setup_seconds):.3f} ms")


### Prox-convex

In [ ]:
import time
scp_started = time.perf_counter()
prox_results_sc = prox_convex(copy.deepcopy(params_sc), cvx_prb, fcn_dict)
scp_seconds = time.perf_counter() - scp_started - prox_results_sc.get('compilation_seconds', 0.)
print(f"\nTotal SCP call (logging included, compilation excluded): {1000.0 * (scp_seconds):.3f} ms")
print(f"All convex calls (transfers excluded): {1000.0 * (prox_results_sc['total_convex_seconds']):.3f} ms")
print("Convex calls, including rejected/terminal trials:", prox_results_sc['convex_solve_count'])
print("Actually used:", prox_results_sc['solvers_used'])
print("Converged:", prox_results_sc['converged'])
print(f"JAX -> NumPy: {1000.0 * (prox_results_sc['total_jax_to_numpy_seconds']):.3f} ms")
print(f"NumPy -> JAX: {1000.0 * (prox_results_sc['total_numpy_to_jax_seconds']):.3f} ms")


In [ ]:
# Totals include rejected AND terminal trials, even if a terminal console row is not printed.
print('Discretization / linearization [ms]:', 1000.0 * (prox_results_sc.get('total_discretization_seconds', float('nan'))))
print('Convex calls, transfers excluded [ms]:', 1000.0 * (prox_results_sc['total_convex_seconds']))
print('JAX -> NumPy [ms]:', 1000.0 * (prox_results_sc['total_jax_to_numpy_seconds']))
print('NumPy -> JAX [ms]:', 1000.0 * (prox_results_sc['total_numpy_to_jax_seconds']))
print('Final output transfer [ms]:', 1000.0 * (prox_results_sc['final_jax_to_numpy_seconds']))
print('Flight time:', float(prox_results_sc['S_new'].sum()))


### Dump results

In [ ]:
# with open('prox_results_sc.pickle', 'wb') as out_f:
#     out_f.write(pickle.dumps(prox_results_sc))

In [ ]:
# with open('prox_results_sc.pickle', 'rb') as f:
#     prox_results_sc = pickle.load(f)      

In [ ]:
# with open('results.pickle', 'wb') as out_f:
#     out_f.write(pickle.dumps(results))
    
# with open('params.pickle', 'wb') as out_f:
#     out_f.write(pickle.dumps(params))

### Simulate

In [ ]:
prox_results = unscale_prox_results(copy.deepcopy(prox_results_sc), params)

params = dynamics(copy.deepcopy(params))

results = RUN(copy.deepcopy(prox_results), 
                 copy.deepcopy(params))

In [ ]:
plot_eventually_demo_figure(results, params, savepath='Figures/eventually_aug_all')

In [ ]:
save_individual_eventually_figures(results, params, save_dir='Figures/eventually_aug')